In [ ]:
# PIPELINE DEPENDENCY INSTALLATION (LOCKED VERSIONS)

# 1. Kunci Numpy di versi stabil (biar Scipy & kawan-kawan tidak pusing) - Base Scientific Computing 
!pip install numpy==1.26.4

# 2. Mesin Utama (PyTorch CUDA & GPU accelerator)
!pip install torch==2.3.0 torchvision==0.18.0 accelerate==0.30.1

# 3. Model Vision & Pemroses Teks (Vision-Language & Tokenization)
!pip install transformers==4.41.2 tiktoken==0.7.0 einops==0.8.0

# 4. Alat Bedah PDF (Document & Image Parsing)
!pip install pdf2image==1.17.0 pillow==10.3.0 verovio==3.15.0

# 5. Ekosistem LangChain & Integrasi Vector (LangChain & Vector Store Integrations)
!pip install langchain==0.2.5 langchain-community==0.2.5 langchain-qdrant==0.1.1 langchain-ollama==0.1.0 qdrant-client==1.9.0

!pip install pypdf

In [ ]:
import glob
import json
import os
import tempfile
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.vectorstores import Qdrant
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pdf2image import convert_from_path
from qdrant_client import QdrantClient
import torch
from transformers import AutoModel, AutoTokenizer

# Target dedicated GPU for OCR processing
os.environ['CUDA_VISIBLE_DEVICES'] = os.getenv('CUDA_VISIBLE_DEVICES', '1')

# --- CONFIGURATION ---
OLLAMA_BASE_URL = os.getenv('OLLAMA_BASE_URL', 'http://127.0.0.1:11434')
QDRANT_URL = os.getenv('QDRANT_URL', 'http://127.0.0.1:6333')
DATA_DIR = './data_awal'
TEMP_DIR = os.getenv('TEMP_DIR', '/tmp/emi_ocr_cache')
RESUME_LOG_FILE = 'resume_log.json'

print('Initializing GOT-OCR 2.0 Model...')
tokenizer = AutoTokenizer.from_pretrained(
    'stepfun-ai/GOT-OCR2_0', trust_remote_code=True
)
model = AutoModel.from_pretrained(
    'stepfun-ai/GOT-OCR2_0',
    trust_remote_code=True,
    torch_dtype=torch.float16,
    device_map='cuda',
).eval()

print('Connecting to Ollama Embedding & Qdrant Vector Store...')
embeddings = OllamaEmbeddings(
    model='bge-m3:latest', base_url=OLLAMA_BASE_URL
)
qdrant_client = QdrantClient(url=QDRANT_URL, timeout=60)
vector_store = Qdrant(
    client=qdrant_client,
    collection_name='emi_knowledge',
    embeddings=embeddings,
)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200
)


def load_resume_log():
  if os.path.exists(RESUME_LOG_FILE):
    with open(RESUME_LOG_FILE, 'r') as f:
      return json.load(f)
  return []


def save_resume_log(pdf_name, log_data):
  log_data.append(pdf_name)
  with open(RESUME_LOG_FILE, 'w') as f:
    json.dump(log_data, f)
  print(f'   [CHECKPOINT] {pdf_name} successfully indexed.')


def run_got_ocr(image_paths):
  results = []
  for i, img_path in enumerate(image_paths):
    print(f'   [GOT-OCR] Processing Page {i+1}...')
    try:
      res = model.chat(tokenizer, img_path, ocr_type='format')
      if res:
        results.append(f'\n\n{res}\n\n')
    except Exception as e:
      print(f'   [ERROR] Failed to process page {i+1}: {e}')
  return ''.join(results)


def main():
  print('Starting Document Processing & Vector Injection Pipeline...')

  pdf_files = glob.glob(os.path.join(DATA_DIR, '*.pdf'))
  os.makedirs(TEMP_DIR, exist_ok=True)
  resume_log = load_resume_log()

  for file_path in pdf_files:
    file_name = os.path.basename(file_path)

    if file_name in resume_log:
      print(f'\n[SKIP] {file_name} already indexed.')
      continue

    print(f'\nProcessing: {file_name}')
    loader = PyPDFLoader(file_path)
    pdf_pages = loader.load()
    sample_text = pdf_pages[0].page_content.strip() if pdf_pages else ''

    docs_to_chunk = []

    # Detect scanned vs digital PDF
    if len(sample_text) < 50:
      print('   [SCAN DETECTED] Routing to GOT-OCR 2.0 pipeline...')
      with tempfile.TemporaryDirectory(dir=TEMP_DIR) as temp_path:
        images = convert_from_path(
            file_path,
            dpi=200,
            output_folder=temp_path,
            fmt='jpeg',
            paths_only=True,
        )
        full_text = run_got_ocr(images)

        if full_text.strip():
          docs_to_chunk.append(
              Document(page_content=full_text, metadata={'source': file_path})
          )
        else:
          print('   [WARNING] No text extracted by OCR.')
    else:
      print('   [DIGITAL PDF] Direct extraction.')
      docs_to_chunk.extend(pdf_pages)

    # Ingestion into Vector Store
    if docs_to_chunk:
      print(f'   Chunking documents for {file_name}...')
      chunks = text_splitter.split_documents(docs_to_chunk)

      print(f'   Injecting {len(chunks)} vectors into Qdrant...')
      vector_store.add_documents(documents=chunks)

      save_resume_log(file_name, resume_log)

  print('\nPipeline execution completed successfully.')


if __name__ == '__main__':
  main()